In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, '.')
from utils import DATA_DIR

# ── Paths ──────────────────────────────────────────────────────────────────
IATI_PATH     = DATA_DIR / 'iati-drc-cleaned.csv'
CONFLICT_PATH = DATA_DIR / 'conflict_dat_cleaned.csv'
OUT_PATH      = DATA_DIR / 'violence_aid_merged.csv'

CUTOFF_START = pd.Timestamp('2021-01-01')
CUTOFF_END   = pd.Timestamp('2026-12-31')


# ── Helper functions ───────────────────────────────────────────────────────

def build_balanced_grid(conflict, iati):
    """Return a full admin2 × month cartesian grid spanning 2021-01 to 2026-12."""
    all_admin2 = pd.Index(
        pd.concat([
            conflict['town_admin2'].rename('admin2_name'),
            iati['admin2_name']
        ]).dropna().unique(),
        name='admin2_name'
    )
    all_months = pd.period_range(start='2021-01', end='2026-12', freq='M')
    grid = pd.MultiIndex.from_product(
        [all_admin2, all_months], names=['admin2_name', 'year_month']
    ).to_frame(index=False)
    print(f'Balanced grid: {len(grid):,} rows '
          f'({all_admin2.nunique()} admin2 units × {len(all_months)} months)')
    return grid, all_months


def aggregate_conflict(conflict):
    """Pass through pre-aggregated ACLED conflict data, renaming key column."""
    return conflict.rename(columns={'town_admin2': 'admin2_name'})[
        ['admin2_name', 'year_month', 'violent_incidents', 'total_deaths']
    ]


def build_aid_monthly(iati_valid, all_months):
    """Vectorised fractional spend allocation.

    Each project's spend is allocated across calendar months proportional
    to the number of days active in each month. Replaces the former
    iterrows + append loop.
    """
    # Month table with period start/end timestamps for vectorised overlap calc
    month_df = pd.DataFrame({
        'year_month':   all_months,
        'month_start':  [p.start_time for p in all_months],
        'month_end':    [p.end_time   for p in all_months],
    })

    # Cross-join: every project × every month, then keep only overlapping months
    crossed = iati_valid.assign(_key=1).merge(month_df.assign(_key=1), on='_key').drop(columns='_key')
    mask = (crossed['day_start'] <= crossed['month_end']) & (crossed['day_end'] >= crossed['month_start'])
    crossed = crossed[mask].copy()

    # Compute fractional spend for each (project, month) pair
    crossed['overlap_start']    = crossed[['day_start', 'month_start']].max(axis=1)
    crossed['overlap_end']      = crossed[['day_end',   'month_end']  ].min(axis=1)
    crossed['active_days']      = (crossed['overlap_end'] - crossed['overlap_start']).dt.days + 1
    crossed['fractional_spend'] = (crossed['active_days'] / crossed['total_days']) * crossed['spend']

    return (
        crossed
        .groupby(['admin2_name', 'year_month'])
        .agg(num_aid_projects=('aid', 'nunique'), total_aid_spend=('fractional_spend', 'sum'))
        .reset_index()
    )


def merge_onto_grid(grid, conflict_monthly, aid_monthly):
    """Left-merge conflict and aid aggregations onto the balanced grid."""
    print(f'Pre-merge  — grid rows: {len(grid):,}')
    panel = (
        grid
        .merge(conflict_monthly, on=['admin2_name', 'year_month'], how='left')
        .merge(aid_monthly,      on=['admin2_name', 'year_month'], how='left')
    )
    print(f'Post-merge — panel rows: {len(panel):,}')
    assert len(panel) == len(grid), 'Row count changed — check for duplicate keys in aggregations'

    # True zeros for months with no observations
    panel['violent_incidents'] = panel['violent_incidents'].fillna(0).astype(int)
    panel['total_deaths']      = panel['total_deaths'].fillna(0).astype(int)
    panel['num_aid_projects']  = panel['num_aid_projects'].fillna(0).astype(int)
    panel['total_aid_spend']   = panel['total_aid_spend'].fillna(0)
    return panel.sort_values(['admin2_name', 'year_month']).reset_index(drop=True)


# ── Load data ──────────────────────────────────────────────────────────────
iati     = pd.read_csv(IATI_PATH)
conflict = pd.read_csv(CONFLICT_PATH)
print(f'IATI rows loaded    : {len(iati):,}')
print(f'Conflict rows loaded: {len(conflict):,}')

# ── Parse & filter dates ───────────────────────────────────────────────────
iati['day_start'] = pd.to_datetime(iati['day_start'])
iati['day_end']   = pd.to_datetime(iati['day_end'])

conflict['date_start'] = pd.to_datetime(conflict['date_start'])
conflict['year_month'] = pd.PeriodIndex(conflict['year_month'], freq='M')
# Already filtered to 2021-2026 in notebook 05; assert to be safe
conflict = conflict[conflict['date_start'].dt.year.between(2021, 2026)]

iati = iati[
    (iati['day_start'] <= CUTOFF_END) &
    (iati['day_end']   >= CUTOFF_START)
]
print(f'IATI after date filter    : {len(iati):,}')
print(f'Conflict after date filter: {len(conflict):,}')

# ── Build balanced grid ────────────────────────────────────────────────────
grid, all_months = build_balanced_grid(conflict, iati)

# ── Aggregate conflict ─────────────────────────────────────────────────────
conflict_monthly = aggregate_conflict(conflict)

# ── Aggregate aid (vectorised — no iterrows) ───────────────────────────────
iati_valid = iati.dropna(subset=['day_start', 'day_end', 'spend', 'admin2_name']).copy()
iati_valid['total_days'] = (iati_valid['day_end'] - iati_valid['day_start']).dt.days.clip(lower=1)
aid_monthly = build_aid_monthly(iati_valid, all_months)
print(f'Aid monthly rows: {len(aid_monthly):,}')

# ── Merge & save ───────────────────────────────────────────────────────────
panel = merge_onto_grid(grid, conflict_monthly, aid_monthly)

print(f'\nPanel summary:')
print(f'  Unique admin2 units : {panel["admin2_name"].nunique()}')
print(f'  Unique months       : {panel["year_month"].nunique()}')
print(f'  Date range          : {panel["year_month"].min()} – {panel["year_month"].max()}')
print(f'  Rows with conflict  : {(panel["violent_incidents"] > 0).sum():,}')
print(f'  Rows with aid       : {(panel["num_aid_projects"]  > 0).sum():,}')

panel.to_csv(OUT_PATH, index=False)
print(f'\nSaved {len(panel):,} rows → {OUT_PATH}')


IATI rows loaded    : 4,917
Conflict rows loaded: 10,595
IATI after date filter    : 3,470
Conflict after date filter: 10,595
Balanced grid: 11,808 rows (164 admin2 units × 72 months)


Aid monthly rows: 8,302
Pre-merge  — grid rows: 11,808
Post-merge — panel rows: 11,808

Panel summary:
  Unique admin2 units : 164
  Unique months       : 72
  Date range          : 2021-01 – 2026-12
  Rows with conflict  : 2,105
  Rows with aid       : 8,302

Saved 11,808 rows → /Users/jackzipper/QSS20/final_project/final_project_data/violence_aid_merged.csv
